# Which Language Is This Short Text In? Character-Level Language Identification for Six Romance-Family Languages

**Name:** Arturo Ramos
**Dataset:** *A massively parallel corpus: the Bible in 100 languages* (Christodoulopoulos & Steedman, 2015), GitHub repository `christos-c/bible-corpus`, CC0 1.0, pinned to commit `44e5fca1bfb369a5da2ee23ebc6f421c88489c5c` — https://github.com/christos-c/bible-corpus

**Deep learning task.** This is a **text (character-sequence) modeling task solved with a Transformer**. Given a short snippet of 10 to 64 characters, the model predicts its language among six closely related languages: Spanish, Portuguese, Italian, French, Romanian and Latin. Short snippets are the hard and realistic case: the same problem appears when a multilingual app has to decide the language of a short social post before translating it. The baseline is a small character-level Transformer encoder; the controlled experiment replaces only the encoder **layer type** with a bidirectional GRU (a recurrent neural network) and keeps everything else identical.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required for deterministic cuBLAS kernels

import random
import re
import time
import unicodedata
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, f1_score

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
# the fused "flash" and "memory-efficient" attention kernels are not deterministic on CUDA; use the math kernel
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("data") / "raw"
FIG_DIR = Path("figures")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

## 2. Data Acquisition

The six translations are downloaded from the public corpus repository at a fixed commit, so the exact same files are used on every run. Some of these translations are still under copyright, so the files are not redistributed in this repository; the cell below fetches them (about 35 MB) and caches them in `data/raw/`.

In [ ]:
CORPUS_COMMIT = "44e5fca1bfb369a5da2ee23ebc6f421c88489c5c"
LANGUAGES = {"Spanish": "es", "Portuguese": "pt", "Italian": "it", "French": "fr", "Romanian": "ro", "Latin": "la"}
LANG_NAMES = {code: name for name, code in LANGUAGES.items()}
LABELS = ["es", "pt", "it", "fr", "ro", "la"]


def download_corpus_file(name: str) -> Path:
    """Download one language file from the pinned corpus commit (skipped if already cached)."""
    path = DATA_DIR / f"{name}.xml"
    if not path.exists():
        url = f"https://raw.githubusercontent.com/christos-c/bible-corpus/{CORPUS_COMMIT}/bibles/{name}.xml"
        urllib.request.urlretrieve(url, path)
    return path


for name in LANGUAGES:
    p = download_corpus_file(name)
    print(f"{name:11s} {p.stat().st_size / 1e6:5.1f} MB  {p}")

## 3. Load and Inspect the Data

In [ ]:
def load_verses(name: str, code: str) -> pd.DataFrame:
    """Parse one CES-XML Bible file into a table with one row per verse segment."""
    root = ET.parse(DATA_DIR / f"{name}.xml").getroot()
    rows = []
    for seg in root.iter("seg"):
        _, book, chapter, verse = seg.get("id").split(".")[:4]
        rows.append({"lang": code, "book": book, "chapter": f"{book}.{chapter}", "verse_id": f"{book}.{chapter}.{verse}",
                     "raw_text": (seg.text or "").strip()})
    return pd.DataFrame(rows)


raw = pd.concat([load_verses(n, c) for n, c in LANGUAGES.items()], ignore_index=True)
print("shape:", raw.shape)
raw.head()

In [ ]:
overview = raw.assign(empty=raw["raw_text"].eq(""), length=raw["raw_text"].str.len()).groupby("lang").agg(
    verses=("verse_id", "size"), empty_segments=("empty", "sum"), books=("book", "nunique"),
    median_chars=("length", "median"))
overview.loc[LABELS]

In [ ]:
# One representative sample per language: the same verse (John 11:35 and Genesis 1:3) in all six languages
samples = raw[raw["verse_id"].isin(["JOH.11.35", "GEN.1.3"])].pivot(index="lang", columns="verse_id", values="raw_text")
samples.loc[LABELS]

In [ ]:
# Character-level quirks that a model could exploit instead of learning the language
by_lang = raw[raw["raw_text"] != ""].groupby("lang")["raw_text"]
quirks = pd.DataFrame({
    "share with backtick `": by_lang.apply(lambda s: s.str.contains("`", regex=False).mean()),
    "verses with U+FFFD": by_lang.apply(lambda s: int(s.str.contains("\ufffd", regex=False).sum())),
    "verses with soft hyphen": by_lang.apply(lambda s: int(s.str.contains("\u00ad", regex=False).sum())),
    "share starting lowercase": by_lang.apply(lambda s: s.str[0].str.islower().mean()),
}).loc[LABELS]
quirks.round(3)

In [ ]:
# The Unicode replacement character (U+FFFD) in the Portuguese file: which words contain it?
pt_text = " ".join(raw.loc[raw["lang"] == "pt", "raw_text"])
print("U+FFFD occurrences:", pt_text.count("\ufffd"), "| correct 'à' occurrences:", pt_text.count("à"))
print(Counter(re.findall(r"\S*\ufffd\S*", pt_text)).most_common(8))

**Data quality and preprocessing considerations**

- The six files give 186,909 verse segments. Spanish has 31,100 verses, Portuguese, French and Romanian 31,102, and Italian and Latin more (31,292 and 31,211) because their source editions number some verses differently. The verse counts therefore do not align perfectly, which matters only for how the data is split (below).
- There are empty segments (10 in Portuguese and 12 in Italian); they carry no text and are removed.
- Several formatting artifacts are tied to a single language and could let a model identify the language without any linguistic knowledge:
  - the backtick (`` ` ``) appears in 72.4 % of French verses (used as an apostrophe) and 20.5 % of Romanian verses (used as a closing quotation mark), and in no other language;
  - the Portuguese file has an encoding error: 2,402 verses contain the Unicode replacement character (U+FFFD) where the text should have the crase *à*; the file contains not a single correct *à*, and the affected words are *à*, *às* and *àquele(s)* in almost every case (a handful of words such as *não* or *há* lost a different accented letter);
  - 25 Spanish verses contain an invisible soft hyphen (U+00AD);
  - 96.0 % of Latin verses start in lowercase, against 9–16 % in the other languages.
- All of these are neutralized by normalization: the replacement character in Portuguese is restored as *à*, backticks become apostrophes, soft hyphens are removed and all text is lowercased. The model then has to rely on real letters and words.
- Verses are long (medians of 100 to 122 characters) but real short posts are not; the model is therefore trained and evaluated on snippets of 10 to 64 characters cut out of the verses.